In [11]:
from cvxpy import Variable, Problem, Maximize, trace, real, partial_trace
from qiskit.quantum_info import random_statevector, DensityMatrix
from numpy import matrix, eye, zeros, ones, concatenate, array
from math import sqrt

In [2]:
def vec(A):
    return matrix(concatenate(array(A), axis = None)).conjugate()

In [5]:
n = 1
p = 0.01

X = Variable((2**(2*n), 2**(2*n)), hermitian=True)

I = eye(2**n)
O = ones((2**n, 2**n))
rho0 = matrix([[1.0, 0.0], [0.0, 0.0]])
rho1 = matrix([[0.0, 0.0], [0.0, 1.0]])
E = [sqrt(p)*I, sqrt(1-p)*(O-I)]

In [8]:
C_0 = [vec(rho0@E[0]), vec(rho0@E[1])]
C_1 = [vec(rho1@E[0]), vec(rho1@E[1])]

C0 = C_0[0].getH().dot(C_0[0]) + C_0[1].getH().dot(C_0[1])
C1 = C_1[0].getH().dot(C_1[0]) + C_1[1].getH().dot(C_1[1])

In [9]:
constraints = [X >> 0]
constraints += [partial_trace(X, (2, 2), 1) == I]

prob = Problem(Maximize(real(trace(X@C0))), constraints)
print(1-prob.solve())

X.value

0.010000035257128448


array([[3.58913781e-08+0.j, 0.00000000e+00+0.j, 0.00000000e+00+0.j,
        0.00000000e+00+0.j],
       [0.00000000e+00+0.j, 9.99999964e-01+0.j, 0.00000000e+00+0.j,
        0.00000000e+00+0.j],
       [0.00000000e+00+0.j, 0.00000000e+00+0.j, 5.00000000e-01+0.j,
        0.00000000e+00+0.j],
       [0.00000000e+00+0.j, 0.00000000e+00+0.j, 0.00000000e+00+0.j,
        5.00000000e-01+0.j]])

In [88]:
constraints = [X >> 0]
constraints += [partial_trace(X, (2, 2), 1) == I]

prob = Problem(Maximize(real(trace(X@C1))), constraints)
print(1-prob.solve())

X.value

0.010000035257129447


array([[5.00000000e-01+0.j, 0.00000000e+00+0.j, 0.00000000e+00+0.j,
        0.00000000e+00+0.j],
       [0.00000000e+00+0.j, 5.00000000e-01+0.j, 0.00000000e+00+0.j,
        0.00000000e+00+0.j],
       [0.00000000e+00+0.j, 0.00000000e+00+0.j, 9.99999964e-01+0.j,
        0.00000000e+00+0.j],
       [0.00000000e+00+0.j, 0.00000000e+00+0.j, 0.00000000e+00+0.j,
        3.58913800e-08+0.j]])

In [100]:
from scipy.linalg import sqrtm

R0 = matrix([[0, 1], [sqrt(0.5), sqrt(0.5)]])
print(trace(sqrtm(rho0.dot(rho0).dot(rho0))).value**2)

S0 = E[0].dot(rho0).dot(E[0])+E[1].dot(rho0).dot(E[1])
print(trace(sqrtm(rho0.dot(S0).dot(rho0))).value**2)

S0 = R0.dot(S0).dot(R0)
print(trace(sqrtm(rho0.dot(S0).dot(rho0))).value**2)

R1 = matrix([[sqrt(0.5), sqrt(0.5)], [1, 0]])
print(trace(sqrtm(rho0.dot(rho0).dot(rho0))).value**2)

S1 = E[0].dot(rho0).dot(E[0])+E[1].dot(rho0).dot(E[1])
print(trace(sqrtm(rho0.dot(S1).dot(rho0))).value**2)

S1 = R1.dot(S1).dot(R1)
print(trace(sqrtm(rho0.dot(S1).dot(rho0))).value**2)

1.0
0.010000000000000002
0.700035713374682
1.0
0.010000000000000002
0.7050357133746821


In [90]:
X = Variable((n+1,n+1), symmetric=True)

constraints = [X >> 0]
constraints += [X[0,0] == 1]
# for i in range(1,n+1):
#     for j in range(1,n+1):
#         if E[i-1,j-1] == 1:
#             constraints += [X[i,j] == 0]
for i in range(1,n+1):
    constraints += [X[0,i] == X[i,i]]
#     constraints += [X[i,i] == P_list[i-1]]

# a = zeros((n+1, n+1))
a = C@X

prob = Problem(Maximize(trace(a+a)-1), constraints)
prob.solve()

2.9999778509359434